# TP 4 — Partie 5 : batch ou streaming ?

Le streaming n'est pas toujours le bon choix. Une requête qui tourne en permanence occupe des ressources jour et nuit, crée des fichiers à chaque micro-lot et demande plus de surveillance. La bonne question est : **de quel délai le métier a-t-il besoin ?**

On calcule ici le même indicateur, **le nombre d'incidents par transporteur**, de trois façons, et on mesure ce que coûte chacune :

| Façon | Principe | Lancement |
|---|---|---|
| **Batch** | relire tout le topic et tout recalculer | à heure fixe, par exemple chaque nuit |
| **Flux continu** | un micro-lot toutes les 5 secondes | une requête qui ne s'arrête jamais |
| **Flux déclenché** (`availableNow`) | traiter les nouveaux messages depuis la dernière fois, puis s'arrêter | à heure fixe, par exemple chaque heure : depuis le notebook, puis par Airflow (étape 5) |

> **⚠️ Un seul notebook actif à la fois**, et les profils `streaming` et `orchestration` démarrés (Kafka, Kafka UI et Airflow). Arrêtez Trino, Superset et la supervision pour laisser la mémoire à Airflow :
> ```
> docker compose --profile full --profile observabilite stop
> docker compose --profile streaming --profile orchestration up -d
> ```

In [ ]:
import time
from datetime import datetime, timezone


def fin_du_lot(p):
    """Heure (en secondes) à laquelle un micro-lot s'est terminé, d'après son rapport d'avancement."""
    debut = datetime.fromisoformat(p["timestamp"].replace("Z", "+00:00")).timestamp()
    return debut + p["durationMs"]["triggerExecution"] / 1000

from pyspark.sql import SparkSession, functions as F

from flux import (CHECKPOINTS, KAFKA, SCHEMA_EVENEMENT, TOPIC, a_completer, arreter_flux, creer_topic,
                  fin_du_rejeu, lancer_dag, rejouer, vider_s3)

spark = (SparkSession.builder.appName("tp4-batch-streaming")
         .config("spark.ui.showConsoleProgress", "false")   # pas de barres de progression
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")     # n'affiche que les erreurs de Spark
arreter_flux(spark)
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
for t in ["incidents_continu", "incidents_declenche", "incidents_horaire", "passages_incidents"]:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse.gold.{t} PURGE")
for c in ["incidents_continu", "incidents_declenche", "incidents_airflow"]:
    vider_s3(f"{CHECKPOINTS}/{c}")
creer_topic(TOPIC)
mesures = {}
print("prêt")

## Étape 1 — Un seul calcul pour les trois façons

Écrivez la fonction `incidents(evenements)` : à partir d'un DataFrame d'événements (colonnes du schéma `SCHEMA_EVENEMENT`), elle garde les incidents (`categorie = 'incident'`) et compte leur nombre par `transporteur`, dans une colonne `incidents`.

La même fonction servira en batch et en streaming : c'est l'intérêt de Spark Structured Streaming.

In [ ]:
def lire_kafka(en_flux):
    lecteur = spark.readStream if en_flux else spark.read
    df = (lecteur.format("kafka").option("kafka.bootstrap.servers", KAFKA)
          .option("subscribe", TOPIC).option("startingOffsets", "earliest"))
    if not en_flux:
        df = df.option("endingOffsets", "latest")
    return (df.load().select(F.from_json(F.col("value").cast("string"), SCHEMA_EVENEMENT).alias("e"))
            .select("e.*"))


def incidents(evenements):
    return a_completer("incidents")  # À COMPLÉTER

## Étape 2 — Le batch

On envoie le mois de mars 2018, puis on calcule l'indicateur en batch, en mesurant la durée.

In [ ]:
n_mars = rejouer("--depuis", "2018-03-01", "--jusqu-a", "2018-04-01", "--sans-pause")

debut = time.time()
resultat_batch = incidents(lire_kafka(en_flux=False))
resultat_batch.orderBy(F.desc("incidents")).show()
mesures["batch"] = time.time() - debut
print(f"durée du calcul batch : {mesures['batch']:.1f} s")

## Étape 3 — Le flux continu

La même fonction, sur `lire_kafka(en_flux=True)`, avec un micro-lot toutes les 5 secondes. Le résultat est une agrégation qui change à chaque micro-lot : on l'écrit en mode `complete`, qui remplace la table entière à chaque fois.

Pendant que la requête tourne, on rejoue les deux premières semaines d'avril à un jour par seconde, et on regarde le total d'incidents évoluer.

In [ ]:
continu = (incidents(lire_kafka(en_flux=True)).writeStream
           .outputMode("complete")
           .trigger(processingTime="5 seconds")
           .option("checkpointLocation", f"{CHECKPOINTS}/incidents_continu")
           .toTable("lakehouse.gold.incidents_continu"))
continu.processAllAvailable()                 # rattrape d'abord le mois de mars

rejeu = rejouer("--depuis", "2018-04-01", "--jusqu-a", "2018-04-15", "--jours-par-seconde", 1,
                attendre=False)
debut = time.time()
dernier_affichage = debut
while rejeu.poll() is None:
    time.sleep(0.5)
    if time.time() - dernier_affichage >= 4:
        dernier_affichage = time.time()
        total = spark.table("lakehouse.gold.incidents_continu").agg(F.sum("incidents")).first()[0]
        print(f"{time.time() - debut:5.0f} s : {total} incidents dans la table")
fin_envoi = time.time()                       # le dernier message vient d'être envoyé
n_avril = fin_du_rejeu(rejeu)

Le rejeu est terminé. Combien de temps faut-il pour que la table contienne **tous** les incidents envoyés ?

In [ ]:
attendu = incidents(lire_kafka(en_flux=False)).agg(F.sum("incidents")).first()[0]
while (spark.table("lakehouse.gold.incidents_continu").agg(F.sum("incidents")).first()[0] != attendu
       and time.time() - fin_envoi < 120):
    time.sleep(0.5)
lots = [p for p in continu.recentProgress if p["numInputRows"] > 0 and fin_du_lot(p) > debut]
mesures["continu"] = max(fin_du_lot(p) for p in lots) - fin_envoi
mesures["continu_lot"] = sum(p["durationMs"]["triggerExecution"] for p in lots) / len(lots) / 1000
print(f"incidents attendus : {attendu}")
print(f"le dernier micro-lot les a écrits {mesures['continu']:.1f} s après le dernier envoi")
print(f"pendant le rejeu : {len(lots)} micro-lots, de {mesures['continu_lot']:.1f} s en moyenne")
continu.stop()

> **❓ Question 1.** Combien de temps s'écoule entre l'envoi d'un incident et son arrivée dans la table ? De quoi dépend ce délai ? Pendant ce temps, que fait la requête quand aucun message n'arrive, la nuit par exemple ?

## Étape 4 — Le flux déclenché

Exactement la même requête, avec `.trigger(availableNow=True)` : elle traite tout ce qui est arrivé depuis son dernier passage, écrit le résultat, puis **s'arrête**. Le checkpoint retient où elle en était, et son état (les comptes par transporteur) d'un passage à l'autre.

Complétez la fonction `passage()`, qui lance la requête sur la table `lakehouse.gold.incidents_declenche`, avec le checkpoint `f"{CHECKPOINTS}/incidents_declenche"`, et attend sa fin (`.awaitTermination()`). Elle renvoie la requête.

In [ ]:
def passage():
    return a_completer("passage")  # À COMPLÉTER


for nom, depuis, jusqua in [("1er passage (mars et début avril)", None, None),
                            ("2e passage (après l'arrivée de la fin avril)", "2018-04-15", "2018-05-01")]:
    if depuis:
        rejouer("--depuis", depuis, "--jusqu-a", jusqua, "--sans-pause")
    debut = time.time()
    requete = passage()
    duree = time.time() - debut
    lus = sum(p["numInputRows"] for p in requete.recentProgress)
    print(f"{nom} : {lus} messages traités en {duree:.1f} s\n")
mesures["declenche"] = duree

total_declenche = spark.table("lakehouse.gold.incidents_declenche").agg(F.sum("incidents")).first()[0]
debut = time.time()
total_batch = incidents(lire_kafka(en_flux=False)).agg(F.sum("incidents")).first()[0]
mesures["batch"] = time.time() - debut      # le batch relit maintenant tout le topic
mesures["messages"] = spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA) \
    .option("subscribe", TOPIC).option("startingOffsets", "earliest").option("endingOffsets", "latest").load().count()
print(f"incidents : flux déclenché {total_declenche}, batch {total_batch}")
print(f"le batch a relu les {mesures['messages']} messages du topic en {mesures['batch']:.1f} s")

> **❓ Question 2.** Au 2e passage, combien de messages ont été traités : tout le topic, ou seulement les nouveaux ? Le résultat est-il pourtant complet ? Qu'est-ce qui le permet ?

## Étape 5 — Confier le flux déclenché à Airflow

Lancer le flux déclenché à la main n'a d'intérêt que pour comprendre. En production, on le confie à un orchestrateur, comme le pipeline photo du TP 3. Deux fichiers sont fournis dans votre dossier `tp4` :

- `pipeline/incidents_declenche.py` : la requête de l'étape 4, dans un script, avec sa propre table (`gold.incidents_horaire`) et son propre checkpoint. À chaque passage, il ajoute une ligne à la table `gold.passages_incidents` : l'heure, le nombre de messages traités et la durée ;
- `dag_suivi_incidents.py` : le DAG qui lance ce script.

Complétez le DAG dans l'éditeur de JupyterLab, comme au TP 3 :
1. la planification : toutes les heures (`"@hourly"`) ;
2. une tâche `BashOperator` nommée `incidents_declenche`, qui exécute `cd {TP4}/pipeline && /usr/local/bin/python incidents_declenche.py`.

La cellule suivante copie le DAG dans le dossier que lit Airflow (`mon_travail/tp3/dags/`, défini au TP 3), arrête la session Spark du notebook pour laisser la mémoire à Airflow, puis active le DAG. Dès qu'il est activé, Airflow lance l'exécution de la dernière heure écoulée : on la suit jusqu'au bout. Comptez une à deux minutes.

In [ ]:
import os
import shutil

os.makedirs("../tp3/dags", exist_ok=True)
shutil.copy("dag_suivi_incidents.py", "../tp3/dags/")
spark.stop()                 # le Spark lancé par Airflow a besoin de la mémoire
lancer_dag("suivi_incidents")

De nouveaux événements arrivent : la première quinzaine de mai. Sans attendre l'heure suivante, on déclenche une nouvelle exécution, comme le ferait Airflow à la prochaine heure.

In [ ]:
n_mai = rejouer("--depuis", "2018-05-01", "--jusqu-a", "2018-05-15", "--sans-pause")
lancer_dag("suivi_incidents")

Le journal des passages, écrit par le script à chaque exécution :

In [ ]:
spark = (SparkSession.builder.appName("tp4-batch-streaming")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
passages = spark.table("lakehouse.gold.passages_incidents").orderBy("debut")
passages.show(truncate=False)
mesures["airflow"] = passages.collect()[-1]["duree_s"]

total_airflow = spark.table("lakehouse.gold.incidents_horaire").agg(F.sum("incidents")).first()[0]
total_batch = incidents(lire_kafka(en_flux=False)).agg(F.sum("incidents")).first()[0]
print(f"incidents : flux déclenché par Airflow {total_airflow}, batch {total_batch}")

Ouvrez Airflow (http://localhost:8085), puis le DAG `suivi_incidents` : la vue *Grid* montre au moins deux exécutions, chacune avec sa tâche `incidents_declenche` en vert (une de plus si une heure pile est passée pendant la partie : Airflow lance alors aussi l'exécution de cette heure). Cliquez sur une tâche, puis sur *Logs* : la dernière ligne indique le nombre de messages traités.

> **❓ Question 3.** Combien de messages chaque passage a-t-il traités, et pourquoi le premier en traite-t-il autant ? Si Airflow restait arrêté cinq heures, que ferait le passage suivant ? Pourquoi le DAG interdit-il deux exécutions simultanées (`max_active_runs=1`) ?

## Étape 6 — Ce que chaque façon laisse dans le Lakehouse

Chaque écriture Iceberg crée un snapshot. Comparons les deux tables.

In [ ]:
for t in ["incidents_continu", "incidents_declenche"]:
    n = spark.sql(f"SELECT count(*) FROM lakehouse.gold.{t}.snapshots").first()[0]
    mesures[f"snapshots_{t}"] = n
    print(f"gold.{t:20} : {n} snapshots")

En flux continu, il faut donc entretenir la table : on supprime les anciens snapshots, en ne gardant que le dernier.

In [ ]:
maintenant = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
spark.sql(f"""CALL lakehouse.system.expire_snapshots(table => 'gold.incidents_continu',
              older_than => TIMESTAMP '{maintenant}', retain_last => 1)""").show()
print("snapshots restants :", spark.sql("SELECT count(*) FROM lakehouse.gold.incidents_continu.snapshots").first()[0])

## Étape 7 — Le bilan

Le tableau reprend vos mesures.

In [ ]:
lignes = [
    ("Façon", "Délai avant que la donnée soit visible", "Coût d'un calcul", "Snapshots"),
    ("Batch nocturne", "jusqu'à 24 h (attente de la nuit)",
     f"{mesures['batch']:.1f} s, les {mesures['messages']} messages relus", "1 par nuit"),
    ("Flux continu", f"{mesures['continu']:.1f} s après le dernier envoi",
     f"{mesures['continu_lot']:.1f} s par micro-lot, en permanence",
     f"{mesures['snapshots_incidents_continu']} en moins d'une minute"),
    ("Flux déclenché", "jusqu'à 1 h (si lancé chaque heure)",
     f"{mesures['declenche']:.1f} s, seulement les nouveaux messages",
     f"{mesures['snapshots_incidents_declenche']} (1 par passage)"),
    ("… par Airflow", "jusqu'à 1 h (DAG lancé chaque heure)",
     f"{mesures['airflow']:.1f} s, démarrage de Spark compris", "1 par passage"),
]
largeurs = [max(len(l[i]) for l in lignes) for i in range(4)]
for n, l in enumerate(lignes):
    print("| " + " | ".join(c.ljust(w) for c, w in zip(l, largeurs)) + " |")
    if n == 0:
        print("|" + "|".join("-" * (w + 2) for w in largeurs) + "|")

Le code de la requête ne change pas d'une façon à l'autre : seul le `trigger` décide si elle tourne en continu ou à heure fixe, et Airflow ne fait que la lancer. On peut donc commencer en flux déclenché, puis passer en continu le jour où le métier en a besoin.

> **❓ Question 4.** Pour chacun des besoins suivants, choisissez batch, flux continu ou flux déclenché, et justifiez :
> 1. le bilan mensuel des incidents par transporteur, envoyé à la direction ;
> 2. un SMS envoyé au client dès qu'un incident touche son colis ;
> 3. le tableau de bord du service client, consulté plusieurs fois par jour ;
> 4. l'entraînement chaque semaine d'un modèle qui prédit les retards (TP 1).

## À retenir

- Le même code Spark sert en batch et en streaming ; seuls la lecture (`read` ou `readStream`) et le `trigger` changent.
- Le **flux continu** donne une donnée fraîche en quelques secondes, au prix de ressources réservées en permanence et de nombreux snapshots.
- Le **flux déclenché** (`availableNow`) est un batch incrémental : lancé à heure fixe par un orchestrateur comme Airflow, il ne traite que les nouveaux messages.
- On choisit d'après le **délai dont le métier a besoin**, pas d'après la technologie.